<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Tharusha/KKT_Deshapriya_DSGP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Feature extraction**

In [1]:
!pip install librosa soundfile


In [2]:
import os
import numpy as np
import pandas as pd
import librosa


In [3]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/archive/audio_speech_actors_01-24"


In [4]:
def get_stress_label(filename):
    """
    Extract stress label from RAVDESS filename
    Stressed = Angry (05), Fearful (06)
    Not stressed = Neutral (01), Calm (02), Happy (03)
    """
    emotion_code = int(filename.split("-")[2])

    if emotion_code in [5, 6]:
        return 1  # Stressed
    elif emotion_code in [1, 2, 3]:
        return 0  # Not stressed
    else:
        return None  # Ignore other emotions


In [5]:
def extract_features(file_path):
    # Load audio (use only first 3 seconds for consistency)
    y, sr = librosa.load(file_path, duration=3, offset=0.5)

    # MFCCs (13 coefficients)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs, axis=1)

    # Pitch (fundamental frequency)
    pitches, _ = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    # Energy (RMS)
    energy = np.mean(librosa.feature.rms(y=y))

    # Tempo (speech rate)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    # Combine all features into one vector
    return np.hstack([mfccs_mean, pitch_mean, energy, tempo])


In [ ]:
features = []
labels = []

for actor_folder in os.listdir(DATASET_PATH):
    actor_path = os.path.join(DATASET_PATH, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                label = get_stress_label(file)

                if label is not None:
                    file_path = os.path.join(actor_path, file)
                    feature_vector = extract_features(file_path)

                    features.append(feature_vector)
                    labels.append(label)


In [ ]:
feature_columns = [f"mfcc_{i}" for i in range(1, 14)]
feature_columns += ["pitch", "energy", "tempo"]

df = pd.DataFrame(features, columns=feature_columns)
df["stress"] = labels

df.head()


In [ ]:
df["stress"].value_counts()


In [ ]:
df.to_csv("ravdess_stress_features.csv", index=False)
print("Feature extraction complete. CSV saved!")


In [ ]:
from google.colab import files
files.download("ravdess_stress_features.csv")


**Feature scaling + train/test split**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
X = df.drop("stress", axis=1)
y = df["stress"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
print("Training data mean (approx 0):")
print(X_train_scaled.mean(axis=0)[:5])

print("\nTraining data std (approx 1):")
print(X_train_scaled.std(axis=0)[:5])


**Train ML models (SVM, Random Forest, Logistic Regression)**

In [ ]:
##Import ML models & metrics
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
##Logistic Regression
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_smote, y_train_smote)


In [ ]:
##Evaluate Logistic Regression

y_pred_lr = lr_model.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))


In [ ]:
##Support Vector Machine (SVM)
svm_model = SVC(kernel="rbf", C=1, gamma="scale")

svm_model.fit(X_train_smote, y_train_smote)


In [ ]:
##Evaluate SVM
y_pred_svm = svm_model.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))


In [ ]:
##Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)


In [ ]:
##Evaluate Random Forest
y_pred_rf = rf_model.predict(X_test_scaled)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


In [ ]:
##Compare all models (clean summary)
results = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_rf)
    ]
})

results


In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    roc_auc_score,
    average_precision_score
)


In [ ]:
models = {
    "Logistic Regression": lr_model,
    "SVM": svm_model,
    "Random Forest": rf_model
}

preds = {
    "Logistic Regression": y_pred_lr,
    "SVM": y_pred_svm,
    "Random Forest": y_pred_rf
}

for name in models:
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, preds[name],
        display_labels=["Not Stressed (0)", "Stressed (1)"],
        values_format="d"
    )
    plt.title(f"Confusion Matrix — {name}")
plt.grid(False)
plt.show()


In [ ]:
svm_model_prob = SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=42)
svm_model_prob.fit(X_train_smote, y_train_smote)

# Probabilities for "stressed" class (class 1)
lr_scores = lr_model.predict_proba(X_test_scaled)[:, 1]
svm_scores = svm_model_prob.predict_proba(X_test_scaled)[:, 1]
rf_scores = rf_model.predict_proba(X_test_scaled)[:, 1]



In [ ]:
score_dict = {
    "Logistic Regression": lr_scores,
    "SVM": svm_scores,
    "Random Forest": rf_scores
}

for name, scores in score_dict.items():
    RocCurveDisplay.from_predictions(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.title(f"ROC Curve — {name} (AUC = {auc:.3f})")
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
for name, scores in score_dict.items():
    PrecisionRecallDisplay.from_predictions(y_test, scores)
    ap = average_precision_score(y_test, scores)
    plt.title(f"Precision–Recall Curve — {name} (AP = {ap:.3f})")
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
plt.figure(figsize=(8, 6))

for name, scores in score_dict.items():
    RocCurveDisplay.from_predictions(y_test, scores, name=name)

plt.title("ROC Curves — All Models")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))

for name, scores in score_dict.items():
    PrecisionRecallDisplay.from_predictions(y_test, scores, name=name)

plt.title("Precision–Recall Curves — All Models")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss


In [ ]:
def hinge_loss(y_true, decision_scores):
    """
    y_true: 0/1 labels
    decision_scores: real-valued scores from decision_function
    """
    y = np.where(np.array(y_true) == 1, 1, -1)  # convert to {-1, +1}
    return np.mean(np.maximum(0, 1 - y * decision_scores))

In [ ]:
lr_sgd = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    random_state=42
)

epochs = 30
train_losses_lr, val_losses_lr = [], []

classes = np.array([0, 1])

for epoch in range(epochs):
    lr_sgd.partial_fit(X_train_smote, y_train_smote, classes=classes)

    # Probabilities for log loss
    p_train = lr_sgd.predict_proba(X_train_smote)
    p_val = lr_sgd.predict_proba(X_test_scaled)

    train_losses_lr.append(log_loss(y_train_smote, p_train))
    val_losses_lr.append(log_loss(y_test, p_val))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1, epochs+1), train_losses_lr, marker='o', label="Train Log Loss")
plt.plot(range(1, epochs+1), val_losses_lr, marker='s', label="Val Log Loss")
plt.xlabel("Epoch")
plt.ylabel("Log Loss")
plt.title("Loss Curve — Logistic Regression (SGD, log_loss)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
svm_sgd = SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    random_state=42
)

epochs = 30
train_losses_svm, val_losses_svm = [], []

for epoch in range(epochs):
    svm_sgd.partial_fit(X_train_smote, y_train_smote, classes=classes)

    train_scores = svm_sgd.decision_function(X_train_smote)
    val_scores = svm_sgd.decision_function(X_test_scaled)

    train_losses_svm.append(hinge_loss(y_train_smote, train_scores))
    val_losses_svm.append(hinge_loss(y_test, val_scores))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1, epochs+1), train_losses_svm, marker='o', label="Train Hinge Loss")
plt.plot(range(1, epochs+1), val_losses_svm, marker='s', label="Val Hinge Loss")
plt.xlabel("Epoch")
plt.ylabel("Hinge Loss")
plt.title("Loss Curve — SVM (SGD, hinge)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_ws = RandomForestClassifier(
    n_estimators=1,
    warm_start=True,
    random_state=42
)

tree_steps = list(range(10, 210, 10))  # 10,20,...,200
train_losses_rf, val_losses_rf = [], []

for n_trees in tree_steps:
    rf_ws.set_params(n_estimators=n_trees)
    rf_ws.fit(X_train_smote, y_train_smote)

    p_train = rf_ws.predict_proba(X_train_smote)
    p_val = rf_ws.predict_proba(X_test_scaled)

    train_losses_rf.append(log_loss(y_train_smote, p_train))
    val_losses_rf.append(log_loss(y_test, p_val))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(tree_steps, train_losses_rf, marker='o', label="Train Log Loss")
plt.plot(tree_steps, val_losses_rf, marker='s', label="Val Log Loss")
plt.xlabel("Number of Trees")
plt.ylabel("Log Loss")
plt.title("Loss Curve — Random Forest (log loss vs trees)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()